This notebook simulates the arrival of new transactional data after the initial historical data consolidation.

The incremental dataset is:

- Loaded into the Bronze Layer
- Cleaned and standardized
- Merged into the Silver Layer using Delta Lake MERGE
- Used by the Gold Layer to refresh analytical tables

This process represents the scheduled ETL pipeline.

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
incremental_path = "/Volumes/workspace/default/fmcg_data/incremental/orders_incremental_01.csv"

bronze_path = "/Volumes/workspace/default/fmcg_data/bronze/"
silver_path = "/Volumes/workspace/default/fmcg_data/silver/"

In [0]:
incremental_orders = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(incremental_path)
)

In [0]:
#display(incremental_orders)

print("Rows :", incremental_orders.count())

Rows : 20


In [0]:
# saving as delta table
incremental_orders.write \
.format("delta") \
.mode("overwrite") \
.save(bronze_path + "orders_incremental")

In [0]:
bronze_increment = spark.read.format("delta").load(
bronze_path + "orders_incremental"
)

standardising

In [0]:
# changing the datatypes to ensure schema is same
bronze_increment = (
    bronze_increment
    .withColumn("Order_ID", col("Order_ID").cast("string"))
    .withColumn("Customer_ID", col("Customer_ID").cast("string"))
    .withColumn("Product_ID", col("Product_ID").cast("string"))
    .withColumn("Store_ID", col("Store_ID").cast("string"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Discount", col("Discount").cast("double"))
    .withColumn("Order_Date", col("Order_Date").cast("timestamp"))
    .withColumn("Created_At", col("Created_At").cast("timestamp"))
    .withColumn("Last_Updated", col("Last_Updated").cast("timestamp"))
    .dropDuplicates(["Order_ID"])
)

In [0]:
bronze_increment.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)



reading the original silver dataset of orders for merging new data in it

In [0]:
silver_orders = DeltaTable.forPath(
    spark,
    silver_path + "orders"
)

In [0]:
(
silver_orders.alias("target")
.merge(
    bronze_increment.alias("source"),
    "target.Order_ID = source.Order_ID"
)
.whenMatchedUpdate(
    condition="""
        source.Last_Updated > target.Last_Updated
    """,
    set={
        "Order_Date":"source.Order_Date",
        "Customer_ID":"source.Customer_ID",
        "Product_ID":"source.Product_ID",
        "Store_ID":"source.Store_ID",
        "Quantity":"source.Quantity",
        "Discount":"source.Discount",
        "Payment_Mode":"source.Payment_Mode",
        "Created_At":"source.Created_At",
        "Last_Updated":"source.Last_Updated"
    }
)
.whenNotMatchedInsertAll()
.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
updated_orders = spark.read.format("delta").load(
    silver_path + "orders"
)

# display(updated_orders)

print("Updated Row Count :", updated_orders.count())

Updated Row Count : 17012


In [0]:
print("Incremental Load Completed Successfully!")

Incremental Load Completed Successfully!
